# Week 57, Day 5: Choosing kernels, and decoding with them

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w057/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Who needs a nonlinear decoder?

The populations and the SVM solver are provided. Fill in the kernels (linear: X X' + 1; Gaussian: exp(-gamma |x - x'|²) + 1, with gamma = 1 / number of neurons) and record test accuracy for each population and decoder.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(575)

def svm_dual(K, Y, C=1.0, sweeps=60):
    n = len(Y); a = np.zeros(n); f = np.zeros(n)
    for s in range(sweeps):
        for i in range(n):
            new = min(max(a[i] - (Y[i] * f[i] - 1) / K[i, i], 0.0), C)
            if new != a[i]:
                f += (new - a[i]) * Y[i] * K[:, i]; a[i] = new
    return a

w_stim, w_ctx = np.random.default_rng(5751).normal(size=(2, 20))   # fixed tuning of the neurons

def population(n, mixed, g):
    stim, ctx = g.integers(0, 2, n) * 2 - 1, g.integers(0, 2, n) * 2 - 1
    pure = np.c_[np.outer(stim, w_stim), np.outer(ctx, w_ctx), np.outer(stim, np.ones(5)) + np.outer(ctx, np.ones(5))]
    feats = np.c_[pure, np.outer(stim * ctx, np.ones(15))] if mixed else pure
    return feats + 1.0 * g.normal(size=feats.shape), (stim * ctx).astype(float)

acc = {}
for kind, mixed in [("pure", False), ("mixed", True)]:
    X, Y = population(300, mixed, r); Xt, Yt = population(1000, mixed, r)
    mu, sd = X.mean(0), X.std(0); X, Xt = (X - mu) / sd, (Xt - mu) / sd
    gam = 1.0 / X.shape[1]
    lin = lambda A, B: A @ B.T + 1
    gau = lambda A, B: ...
    for name, k in [("linear", lin), ("kernel", gau)]:
        a = svm_dual(k(X, X), Y)
        acc[kind, name] = ...
        print(f"{kind:5s} selectivity, {name:6s} decoder: test accuracy {acc[kind, name]:.3f}")

In [ ]:
import numpy as np
def test_pure():
    "from pure selectivity, a linear decoder is near chance and a kernel decoder succeeds"
    assert acc["pure", "linear"] < 0.65 and acc["pure", "kernel"] > 0.85
def test_mixed():
    "with mixed selectivity, a linear decoder succeeds"
    assert acc["mixed", "linear"] > 0.85

test_pure()
test_mixed()
print("All checks passed")

### Exercise: Kernel PCA on two rings

Centre the Gaussian kernel matrix of two noisy concentric rings and compute the projections onto its three leading eigenvectors. Measure how well a threshold on each component separates the rings (the best accuracy over thresholds), and compare with the two ordinary principal components.

In [ ]:
n = 300
lab = np.repeat([0, 1], n // 2)
ang = r.uniform(0, 2 * np.pi, n); rad = np.where(lab == 0, 1.0, 2.5) + 0.2 * r.normal(size=n)
Xr = np.c_[rad * np.cos(ang), rad * np.sin(ang)]
d2 = ((Xr[:, None, :] - Xr[None, :, :]) ** 2).sum(-1)
K = np.exp(-0.1 * d2)
H = np.eye(n) - np.ones((n, n)) / n
Kc = ...
evals, evecs = np.linalg.eigh(Kc)
Z_kpca = np.sqrt(evals[-3:][::-1]) * evecs[:, -3:][:, ::-1]      # the three leading components
Xc = Xr - Xr.mean(0); Z_pca = Xc @ np.linalg.svd(Xc, full_matrices=False)[2].T

def best_threshold_acc(z, y):
    return max(max(np.mean((z > t) == y), np.mean((z <= t) == y)) for t in np.unique(z))

kpca_seps = [best_threshold_acc(Z_kpca[:, k], lab) for k in range(3)]
sep_kpca, sep_pca = max(kpca_seps), max(best_threshold_acc(Z_pca[:, k], lab) for k in range(2))
print("kernel PCA, separation by components 1 to 3:", [round(float(v), 3) for v in kpca_seps])
print(f"best kernel component {sep_kpca:.3f}; best ordinary component {sep_pca:.3f}")

In [ ]:
import numpy as np
def test_center():
    "the centred kernel matrix has rows that sum to zero"
    assert np.allclose(Kc.sum(1), 0, atol=1e-8)
def test_separate():
    "one of the three leading kernel principal components separates the rings, while neither ordinary component does"
    assert sep_kpca > 0.95 and sep_pca < 0.7

test_center()
test_separate()
print("All checks passed")

### Exercise: Decoding a timing code

Write smooth(trains, tau), mapping each trial's spike times onto a fine time grid as a sum of Gaussians. The kernel is the inner product of smoothed traces (plus 1). Decode stimulus with the SVM using this kernel, and compare with a linear SVM on spike counts.

In [ ]:
T, grid = 1.0, np.linspace(0, 1.0, 200)

def trial(stim, g):
    n_sp = g.poisson(8)
    centre = 0.3 if stim > 0 else 0.7
    return np.clip(centre + 0.12 * g.normal(size=n_sp), 0, T)

def smooth(trains, tau=0.05):
    return np.array([...])

ys = np.where(r.random(240) < 0.5, -1.0, 1.0); yt_ = np.where(r.random(1000) < 0.5, -1.0, 1.0)
tr_trains = [trial(s, r) for s in ys]; te_trains = [trial(s, r) for s in yt_]
S, St = smooth(tr_trains), smooth(te_trains)
Ktr = S @ S.T / len(grid) + 1
a = svm_dual(Ktr, ys)
acc_spk = np.mean(np.sign((St @ S.T / len(grid) + 1) @ (a * ys)) == yt_)
cnt, cnt_t = np.array([[len(t)] for t in tr_trains], float), np.array([[len(t)] for t in te_trains], float)
m_, s_ = cnt.mean(), cnt.std()
a_c = svm_dual(((cnt - m_) / s_) @ ((cnt - m_) / s_).T + 1, ys)
acc_count = np.mean(np.sign((((cnt_t - m_) / s_) @ ((cnt - m_) / s_).T + 1) @ (a_c * ys)) == yt_)
print(f"spike-count decoder {acc_count:.3f}; spike-train kernel decoder {acc_spk:.3f}")

In [ ]:
import numpy as np
def test_kernel():
    "the spike-train kernel matrix is positive semi-definite"
    assert np.linalg.eigvalsh(Ktr).min() > -1e-6
def test_decode():
    "spike counts decode at chance; the spike-train kernel decodes well"
    assert acc_count < 0.62 and acc_spk > 0.85

test_kernel()
test_decode()
print("All checks passed")